# **INITIAL GITHUB LOADING**

In [1]:
from google.colab import userdata
import sys, os

TOKEN = userdata.get('GITHUB_TOKEN')
USERNAME = "vedika0806"
REPO = "data266-5937"
HW = "hw1"

os.system(f"git clone https://{TOKEN}@github.com/{USERNAME}/{REPO}.git /content/{REPO}")
sys.path.append(f"/content/{REPO}/common")
from colab_bootstrap import setup_repo, get_logger, git_commit_push, git_tag_push

REPO_PATH = f"/content/{REPO}"
HW_PATH = setup_repo(HW, USERNAME, REPO, TOKEN)
logger = get_logger(HW_PATH)
logger.info("=== CUDA session started ===")

INFO:/content/data266-5937/hw1:=== CUDA session started ===


Working directory: /content/data266-5937/hw1


In [ ]:
!nvidia-smi
!nvcc --version
!which nsys nvprof ncu 2>/dev/null

Tue Sep  1 07:50:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
print("=== Current RUN_LOG.txt content (should show all past sessions) ===")
!cat {HW_PATH}/RUN_LOG.txt

print("\n=== Current METRICS.md content (should show your table + conclusion) ===")
!cat {HW_PATH}/METRICS.md

=== Current RUN_LOG.txt content (should show all past sessions) ===
2026-08-31 17:55:15,886 | === Session started for hw1 ===
2026-08-31 18:13:34,241 | Sanity check: logger is working
2026-08-31 19:22:27,359 | Data preprocessing and visualization completed
2026-08-31 19:29:48,417 | Uploading visualization images
2026-08-31 19:59:41,582 | Split fixed: train=531, val=114, test=114, random_state=5937
2026-08-31 20:00:45,135 | HP_ID=3 -> Modified config: {'hidden_layers': [64, 32], 'lr': 0.0003, 'epochs': 30}
2026-08-31 20:02:22,714 | [PyTorch baseline] seed=5937 test_acc=0.7719
2026-08-31 20:02:22,770 | [PyTorch baseline] seed=5938 test_acc=0.7632
2026-08-31 20:02:22,827 | [PyTorch baseline] seed=5939 test_acc=0.6930
2026-08-31 20:02:22,882 | [PyTorch modified HP_ID=3] seed=5937 test_acc=0.6491
2026-08-31 20:02:22,944 | [PyTorch modified HP_ID=3] seed=5938 test_acc=0.6404
2026-08-31 20:02:22,997 | [PyTorch modified HP_ID=3] seed=5939 test_acc=0.6754
2026-08-31 20:02:56,064 | [PyTorch] bas

# **CUDA CODE FOR MATRIX MULTIPLICATION**

In [ ]:
cuda_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define TILE_WIDTH 16

__global__ void matMulKernel(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

void matMulCPU(float *A, float *B, float *C, int N) {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++)
                sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Usage: %s <matrix_size>\n", argv[0]);
        return 1;
    }
    int N = atoi(argv[1]);
    size_t bytes = N * N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C_gpu = (float*)malloc(bytes);
    float *h_C_cpu = (float*)malloc(bytes);

    srand(5937);
    for (int i = 0; i < N * N; i++) {
        h_A[i] = (float)(rand() % 100) / 10.0f;
        h_B[i] = (float)(rand() % 100) / 10.0f;
    }

    clock_t cpu_start = clock();
    matMulCPU(h_A, h_B, h_C_cpu, N);
    clock_t cpu_end = clock();
    double cpu_time_ms = 1000.0 * (cpu_end - cpu_start) / CLOCKS_PER_SEC;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaEvent_t startH2D, stopH2D, startKernel, stopKernel, startD2H, stopD2H;
    cudaEventCreate(&startH2D); cudaEventCreate(&stopH2D);
    cudaEventCreate(&startKernel); cudaEventCreate(&stopKernel);
    cudaEventCreate(&startD2H); cudaEventCreate(&stopD2H);

    cudaEventRecord(startH2D);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(stopH2D);
    cudaEventSynchronize(stopH2D);
    float h2d_ms = 0;
    cudaEventElapsedTime(&h2d_ms, startH2D, stopH2D);

    dim3 blockDim(TILE_WIDTH, TILE_WIDTH);
    dim3 gridDim((N + TILE_WIDTH - 1) / TILE_WIDTH, (N + TILE_WIDTH - 1) / TILE_WIDTH);

    // ---- WARM-UP LAUNCH (NOT TIMED) ----
    // Absorbs CUDA context creation / kernel JIT compilation cost so it
    // doesn't contaminate the timed measurement below.
    matMulKernel<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    // ---- TIMED KERNEL LAUNCH ----
    cudaEventRecord(startKernel);
    matMulKernel<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stopKernel);
    cudaEventSynchronize(stopKernel);
    float kernel_ms = 0;
    cudaEventElapsedTime(&kernel_ms, startKernel, stopKernel);

    cudaEventRecord(startD2H);
    cudaMemcpy(h_C_gpu, d_C, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(stopD2H);
    cudaEventSynchronize(stopD2H);
    float d2h_ms = 0;
    cudaEventElapsedTime(&d2h_ms, startD2H, stopD2H);

    float transfer_ms = h2d_ms + d2h_ms;
    float gpu_end_to_end_ms = h2d_ms + kernel_ms + d2h_ms;
    float speedup = (float)cpu_time_ms / gpu_end_to_end_ms;

    float maxDiff = 0.0f;
    for (int i = 0; i < N * N; i++) {
        float diff = fabsf(h_C_cpu[i] - h_C_gpu[i]);
        if (diff > maxDiff) maxDiff = diff;
    }

    printf("N=%d | CPU: %.3f ms | GPU kernel: %.3f ms | H2D+D2H: %.3f ms | End-to-end GPU: %.3f ms | Speedup: %.3fx | MaxDiff: %f\n",
           N, cpu_time_ms, kernel_ms, transfer_ms, gpu_end_to_end_ms, speedup, maxDiff);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C_gpu); free(h_C_cpu);

    return 0;
}
'''

with open(f"{HW_PATH}/matmul.cu", "w") as f:
    f.write(cuda_code)

print("matmul.cu updated with warm-up launch.")

matmul.cu updated with warm-up launch.


# **PROFILING**

In [ ]:
sizes = [256, 1024, 4096]
results = {}

for N in sizes:
    print(f"=== Running size {N} ===")
    output = !{HW_PATH}/matmul_exe {N}
    output_text = "\n".join(output)
    print(output_text)

    with open(f"{HW_PATH}/RUN_LOG.txt", "a") as f:
        f.write(f"\n=== CUDA matmul run, N={N} ===\n{output_text}\n")

    results[N] = output_text
    logger.info(f"CUDA matmul N={N}: {output_text}")

git_commit_push(REPO_PATH, "hw1: CUDA kernel compiled and timed for N=256,1024,4096")

=== Running size 256 ===


INFO:/content/data266-5937/hw1:CUDA matmul N=256: N=256 | CPU: 22.008 ms | GPU kernel: 0.028 ms | H2D+D2H: 0.442 ms | End-to-end GPU: 0.470 ms | Speedup: 46.821x | MaxDiff: 0.001465


N=256 | CPU: 22.008 ms | GPU kernel: 0.028 ms | H2D+D2H: 0.442 ms | End-to-end GPU: 0.470 ms | Speedup: 46.821x | MaxDiff: 0.001465
=== Running size 1024 ===


INFO:/content/data266-5937/hw1:CUDA matmul N=1024: N=1024 | CPU: 3690.363 ms | GPU kernel: 0.890 ms | H2D+D2H: 5.115 ms | End-to-end GPU: 6.005 ms | Speedup: 614.516x | MaxDiff: 0.001953


N=1024 | CPU: 3690.363 ms | GPU kernel: 0.890 ms | H2D+D2H: 5.115 ms | End-to-end GPU: 6.005 ms | Speedup: 614.516x | MaxDiff: 0.001953
=== Running size 4096 ===


INFO:/content/data266-5937/hw1:CUDA matmul N=4096: N=4096 | CPU: 693369.540 ms | GPU kernel: 58.827 ms | H2D+D2H: 74.463 ms | End-to-end GPU: 133.290 ms | Speedup: 5201.962x | MaxDiff: 0.007812


N=4096 | CPU: 693369.540 ms | GPU kernel: 58.827 ms | H2D+D2H: 74.463 ms | End-to-end GPU: 133.290 ms | Speedup: 5201.962x | MaxDiff: 0.007812
[main 07f784e] hw1: CUDA kernel compiled and timed for N=256,1024,4096
 3 files changed, 23 insertions(+), 19 deletions(-)
 


In [ ]:
!which nvprof
!nvprof --version

/usr/local/cuda/bin/nvprof
nvprof: NVIDIA (R) Cuda command line profiler
Copyright (c) 2012 - 2024 NVIDIA Corporation
Release version 12.8.90 (21)


In [ ]:
!which nsys nvprof ncu nsight-sys 2>/dev/null
!ls /usr/local/cuda/bin/ 2>/dev/null
!apt list --installed 2>/dev/null | grep -i nsight

/usr/local/cuda/bin/nvprof
/usr/local/cuda/bin/ncu
bin2c			 cuda-gdb-python3.11-tui  fatbinary	       nvlink
compute-sanitizer	 cuda-gdb-python3.12-tui  ncu		       nvprof
crt			 cuda-gdb-python3.8-tui   ncu-ui	       nvprune
cudafe++		 cuda-gdb-python3.9-tui   nvcc		       ptxas
cuda-gdb		 cuda-gdbserver		  __nvcc_device_query
cuda-gdb-minimal	 cu++filt		  nvcc.profile
cuda-gdb-python3.10-tui  cuobjdump		  nvdisasm
cuda-nsight-compute-12-8/unknown,now 12.8.1-1 amd64 [installed,upgradable to: 12.8.2-1]
nsight-compute-2025.1.1/unknown,now 2025.1.1.2-1 amd64 [installed,automatic]


In [ ]:
!/usr/local/cuda/bin/nvprof --version

nvprof: NVIDIA (R) Cuda command line profiler
Copyright (c) 2012 - 2024 NVIDIA Corporation
Release version 12.8.90 (21)


In [ ]:
# N=256
!/usr/local/cuda/bin/ncu --set full -o {HW_PATH}/ncu_report_N256 {HW_PATH}/matmul_exe 256 > {HW_PATH}/ncu_output_N256.txt 2>&1
!cat {HW_PATH}/ncu_output_N256.txt

==PROF== Connected to process 10650 (/content/data266-5937/hw1/matmul_exe)
==PROF== Profiling "matMulKernel" - 0: 0%....50%....100% - 49 passes
==PROF== Profiling "matMulKernel" - 1: 0%....50%....100% - 50 passes
N=256 | CPU: 21.863 ms | GPU kernel: 3120.360 ms | H2D+D2H: 0.468 ms | End-to-end GPU: 3120.828 ms | Speedup: 0.007x | MaxDiff: 0.001465
==PROF== Disconnected from process 10650
==PROF== Report: /content/data266-5937/hw1/ncu_report_N256.ncu-rep


In [ ]:
# N=1024
!/usr/local/cuda/bin/ncu --set full -o {HW_PATH}/ncu_report_N1024 {HW_PATH}/matmul_exe 1024 > {HW_PATH}/ncu_output_N1024.txt 2>&1
!cat {HW_PATH}/ncu_output_N1024.txt

==PROF== Connected to process 10901 (/content/data266-5937/hw1/matmul_exe)
==PROF== Profiling "matMulKernel" - 0: 0%....50%....100% - 49 passes
==PROF== Profiling "matMulKernel" - 1: 0%....50%....100% - 49 passes
N=1024 | CPU: 3360.350 ms | GPU kernel: 3476.543 ms | H2D+D2H: 5.016 ms | End-to-end GPU: 3481.559 ms | Speedup: 0.965x | MaxDiff: 0.001953
==PROF== Disconnected from process 10901
==PROF== Report: /content/data266-5937/hw1/ncu_report_N1024.ncu-rep


In [ ]:
# N=4096
!/usr/local/cuda/bin/ncu --set full -o {HW_PATH}/ncu_report_N4096 {HW_PATH}/matmul_exe 4096 > {HW_PATH}/ncu_output_N4096.txt 2>&1
!cat {HW_PATH}/ncu_output_N4096.txt

==PROF== Connected to process 11120 (/content/data266-5937/hw1/matmul_exe)
==PROF== Profiling "matMulKernel" - 0: 0%
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
....50%....100% - 49 passes
==PROF== Profiling "matMulKernel" - 1: 0%....50%....100% - 50 passes
N=4096 | CPU: 663528.643 ms | GPU kernel: 32055.801 ms | H2D+D2H: 73.936 ms | End-to-end GPU: 32129.736 ms | Speedup: 20.652x | MaxDiff: 0.007812
==PROF== Disconnected from process 11120
==PROF== Report: /content/data266-5937/hw1/ncu_report_N4096.ncu-rep


In [ ]:
for N in [256, 1024, 4096]:
    with open(f"{HW_PATH}/ncu_output_N{N}.txt") as nf:
        content = nf.read()
    with open(f"{HW_PATH}/RUN_LOG.txt", "a") as f:
        f.write(f"\n=== Nsight Compute (ncu) profiler output, N={N} ===\n{content}\n")
    logger.info(f"ncu profiling complete for N={N}")

git_commit_push(REPO_PATH, "hw1: Nsight Compute profiler output for N=256, 1024, 4096")

INFO:/content/data266-5937/hw1:ncu profiling complete for N=256
INFO:/content/data266-5937/hw1:ncu profiling complete for N=1024
INFO:/content/data266-5937/hw1:ncu profiling complete for N=4096


[main e7ca1c4] hw1: Nsight Compute profiler output for N=256, 1024, 4096
 7 files changed, 52 insertions(+)
 create mode 100644 hw1/ncu_output_N1024.txt
 create mode 100644 hw1/ncu_output_N256.txt
 create mode 100644 hw1/ncu_output_N4096.txt
 create mode 100644 hw1/ncu_report_N1024.ncu-rep
 create mode 100644 hw1/ncu_report_N256.ncu-rep
 create mode 100644 hw1/ncu_report_N4096.ncu-rep
 


In [ ]:
git_commit_push(REPO_PATH, "hw1: nvprof profiler output for N=256, 1024, 4096")

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
 


In [5]:
!cd {REPO_PATH} && git log --oneline -10

e7ca1c4 (HEAD -> main, origin/main, origin/HEAD) hw1: Nsight Compute profiler output for N=256, 1024, 4096
7fe05f5 hw1: nvprof profiler output for N=256, 1024, 4096
cac603b hw1: nsys profiler output for all matrix sizes
07f784e hw1: CUDA kernel compiled and timed for N=256,1024,4096
7f35621 hw1: CUDA kernel compiled and timed for N=256,1024,4096
9a545f2 neural_networks.ipynb saved to Github
ff62d68 hw1: METRICS.md updated with accuracy table
94e5597 hw1: PyTorch baseline + HP_ID modified, 3-seed runs complete
512851d hw1: TensorFlow baseline + HP_ID modified, 3-seed runs complete
9bac2dd hw1: PyTorch baseline + HP_ID modified, 3-seed runs complete


In [6]:
final_table = """

| Matrix size | CPU (ms)     | GPU kernel (ms) | H2D+D2H (ms) | Speedup   |
|-------------|--------------|------------------|--------------|-----------|
| 256         | 22.008       | 0.028            | 0.442        | 46.821    |
| 1024        | 3690.363     | 0.890            | 5.115        | 614.516   |
| 4096        | 693369.540   | 58.827           | 74.463       | 5201.962  |
"""

with open(f"{HW_PATH}/METRICS.md", "a") as f:
    f.write(final_table)

print(final_table)



| Matrix size | CPU (ms)     | GPU kernel (ms) | H2D+D2H (ms) | Speedup   |
|-------------|--------------|------------------|--------------|-----------|
| 256         | 22.008       | 0.028            | 0.442        | 46.821    |
| 1024        | 3690.363     | 0.890            | 5.115        | 614.516   |
| 4096        | 693369.540   | 58.827           | 74.463       | 5201.962  |



In [7]:
print("=== METRICS.md ===")
!cat {HW_PATH}/METRICS.md

=== METRICS.md ===

## HW1 Results — HP_ID=3 (Modified config: {'hidden_layers': [64, 32], 'lr': 0.0003, 'epochs': 30})

| Framework  | Model    | Mean Test Acc | Std Test Acc |
|------------|----------|----------------|----------------|
| PyTorch    | Baseline | 0.7427 | 0.0353 |
| PyTorch    | Modified | 0.6550 | 0.0149 |
| TensorFlow | Baseline | 0.7953 | 0.0041 |
| TensorFlow | Modified | 0.7719 | 0.0248 |


| Matrix size | CPU (ms)     | GPU kernel (ms) | H2D+D2H (ms) | Speedup   |
|-------------|--------------|------------------|--------------|-----------|
| 256         | 22.008       | 0.028            | 0.442        | 46.821    |
| 1024        | 3690.363     | 0.890            | 5.115        | 614.516   |
| 4096        | 693369.540   | 58.827           | 74.463       | 5201.962  |


In [8]:
def git_commit_push(repo_path, message):
    subprocess.run(["git", "add", "."], cwd=repo_path)
    result = subprocess.run(["git", "commit", "-m", message], cwd=repo_path,
                             capture_output=True, text=True)
    print(result.stdout, result.stderr)
    subprocess.run(["git", "push"], cwd=repo_path)

In [13]:
import subprocess
git_commit_push(REPO_PATH, "hw1: final CUDA timing table and crossover analysis added to METRICS.md")

[main 598691b] hw1: final CUDA timing table and crossover analysis added to METRICS.md
 3 files changed, 8 insertions(+)
 


In [16]:
logger.info("Analysed matrix multiplication tables and provided conclusion")

INFO:/content/data266-5937/hw1:Analysed matrix multiplication tables and provided conclusion


In [17]:
!cat {HW_PATH}/RUN_LOG.txt

2026-08-31 17:55:15,886 | === Session started for hw1 ===
2026-08-31 18:13:34,241 | Sanity check: logger is working
2026-08-31 19:22:27,359 | Data preprocessing and visualization completed
2026-08-31 19:29:48,417 | Uploading visualization images
2026-08-31 19:59:41,582 | Split fixed: train=531, val=114, test=114, random_state=5937
2026-08-31 20:00:45,135 | HP_ID=3 -> Modified config: {'hidden_layers': [64, 32], 'lr': 0.0003, 'epochs': 30}
2026-08-31 20:02:22,714 | [PyTorch baseline] seed=5937 test_acc=0.7719
2026-08-31 20:02:22,770 | [PyTorch baseline] seed=5938 test_acc=0.7632
2026-08-31 20:02:22,827 | [PyTorch baseline] seed=5939 test_acc=0.6930
2026-08-31 20:02:22,882 | [PyTorch modified HP_ID=3] seed=5937 test_acc=0.6491
2026-08-31 20:02:22,944 | [PyTorch modified HP_ID=3] seed=5938 test_acc=0.6404
2026-08-31 20:02:22,997 | [PyTorch modified HP_ID=3] seed=5939 test_acc=0.6754
2026-08-31 20:02:56,064 | [PyTorch] baseline mean=0.7427 std=0.0353
2026-08-31 20:02:56,069 | [PyTorch] mod

# **CONCLUSION**


---

**Profiler used**: Nsight Compute (ncu) — nsys was not installed in this Colab
environment, and nvprof is unsupported on Ampere-generation GPUs (A100,
compute capability 8.0+).



---
Across all measurements, GPU end to end runtime was more than CPU Baseline even for N = 256: GPU end-to-end time was 0.470 ms versus the CPU's 22.008 ms. As the matrix grows, the gap becomes much bigger.
The CPU time grows with the increase in matrix size, but the same does not happen with the GPU as the kernel time only increases from 0.028 ms to 58.8 ms (about 2100x) even though the actual amount of work increased 64x. This happens because the GPU spread the job across multiple cores unlike CPU on a single core.
Every GPU has some fixed cost no matter the size of the input: kernel launch takes time and moving data to and from GPU takes time too. Data transfer takes more time than actual computation, so the overhead this more even if the matrix size is shrunk.
The GPU end-to-end time becomes beneficial at N=256, however there is no crossover observed.